# matching-pmh: frozen features + sklearn (no PyTorch training)

**Story:** You already exported embeddings from **Site A** (train) and **Site B** (deploy). Same labels, different sensor/site. No image download, no GPU training loop.

This notebook uses **synthetic** feature matrices so it runs anywhere in ~2 minutes.

In [ ]:
!pip install -q "matching-pmh[sklearn]"

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from pmh import PMHMatcher
from pmh.onboarding import preflight_plain_english

rng = np.random.default_rng(0)
n, d = 500, 128
q, _ = np.linalg.qr(rng.standard_normal((d, 12)).astype(np.float32))
x_src = rng.standard_normal((n, d)).astype(np.float32)
y_src = rng.integers(0, 5, n)
nuisance = (x_src @ q) @ q.T
x_tgt = x_src + 1.2 * nuisance + 0.05 * rng.standard_normal((n, d)).astype(np.float32)
y_tgt = y_src.copy()

x_tr, x_te, y_tr, y_te = train_test_split(x_tgt, y_tgt, test_size=0.35, random_state=0)
print(f"Source {x_src.shape}, target pool {x_tr.shape}, target test {x_te.shape}")

In [ ]:
# Baseline: train on source only, test on held-out TARGET rows
clf_b0 = LogisticRegression(max_iter=500)
clf_b0.fit(x_src, y_src)
acc_b0 = accuracy_score(y_te, clf_b0.predict(x_te))
print(f"Target accuracy (baseline, source-only train): {acc_b0:.3f}")

In [ ]:
matcher = PMHMatcher(nuisance="domain_shift", rank=16, seed=0)
matcher.fit(x_src, x_tgt=x_tr)  # adapt using target pool (not test rows)
pf = matcher.artifact_.preflight
print(f"Preflight: {pf} - {preflight_plain_english(pf)}")

pipe = Pipeline([
    ("adapt", matcher),
    ("clf", LogisticRegression(max_iter=500)),
])
pipe.fit(x_src, y_src)
acc_pmh = accuracy_score(y_te, pipe.predict(x_te))
print(f"Target accuracy (PMH adapt + clf):            {acc_pmh:.3f}")

## What next?

1. Replace synthetic arrays with your `.npy` / parquet embeddings.
2. Keep a **target holdout** that never goes into `matcher.fit`.
3. Optional falsification: `compare_arms_sklearn(...)` after basics work.

```bash
pmh-train wizard --non-interactive --stack sklearn
```

Docs: [Gallery tabular](https://github.com/vishalstark512/matching-pmh/blob/main/docs/gallery/tabular.md)